In [0]:
"""
id: source_0
template: source
templateVersion: 1.0.0
name: sale_refund
position:
  x: 0
  y: 0
description:
  text: Load all data from the sale_refund table.
  hash: 7c7e296f
previewCodeHash: 537717fcfe16a81f
previewMode: "1000"
config:
  table_source:
    tableName: hub.`default`.sale_refund
input: []
"""

# generated from the system
from typing import Dict, Any

def _strip_sql_quotes(s):
    if isinstance(s, str) and len(s) >= 2:
        if (s[0] == '"' and s[-1] == '"') or (s[0] == "'" and s[-1] == "'"):
            return s[1:-1]
    return s

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    file_source = config.get("file_source")
    table_source = config.get("table_source")

    if file_source:
        path = file_source.get("path")
        if not path:
            raise ValueError("Source: 'path' is required for file source")
        options = []
        for key, value in file_source.items():
            if key == "path":
                continue
            if key == "headerRows" and isinstance(value, bool):
                options.append(f"{key}=>{1 if value else 0}")
            elif isinstance(value, bool):
                options.append(f'{key}=>{"true" if value else "false"}')
            elif isinstance(value, (int, float)):
                options.append(f"{key}=>{value}")
            else:
                clean = _strip_sql_quotes(str(value))
                if key == "dataAddress" and clean.startswith("!"):
                    clean = clean[1:]
                options.append(f'{key}=>"{clean}"')
        opts = ", ".join(options)
        sql = f'SELECT * FROM read_files("{path}", {opts})' if opts else f'SELECT * FROM read_files("{path}")'
        out = spark.sql(sql)
    elif table_source:
        table_name = table_source.get("tableName")
        if not table_name:
            raise ValueError("Source: 'tableName' is required for table source")
        out = spark.table(table_name)
    else:
        raise ValueError("Source: either 'file_source' or 'table_source' must be configured")

    return {"data": out}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "table_source": {
        "tableName": "hub.`default`.sale_refund"
    }
}
inputs = {}
out = run(config, inputs, spark)
ctx["source_0.data"] = out["data"]

In [0]:
"""
id: sql_1
template: sql
templateVersion: 1.0.0
name: sql_1
position:
  x: 300
  y: 0
description:
  text: Retrieve all data from the sale_refund table.
  hash: 5bf434c7
previewCodeHash: baada51b3cf4d066
previewMode: "1000"
config:
  query: SELECT * FROM sale_refund WHERE customer_id = 101
input:
  - node: source_0
    input_port: data
    output_port: data
"""

# generated from the system
import re
from typing import Any, Dict, List

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    sources: List[Dict[str, str]] = inputs.get("data__sources") or []
    for i, df in enumerate(inputs.get("data") or []):
        if df is not None and i < len(sources):
            df.createOrReplaceTempView(sources[i]["df_name"])
            globals().setdefault("_lb_views", set()).add(sources[i]["df_name"])

    query = config.get("query", "")
    param_names = set(re.findall(r"(?<!:):(\w+)", query))
    if param_names:
        all_widgets = dbutils.widgets.getAll()
        args = {name: all_widgets[name] for name in param_names if name in all_widgets}
        result = spark.sql(query, args=args) if args else spark.sql(query)
    else:
        result = spark.sql(query)
    return {"result": result}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "query": "SELECT * FROM sale_refund WHERE customer_id = 101"
}
inputs = {
    "data": [
        ctx["source_0.data"]
    ],
    "data__sources": [
        {
            "node": "source_0",
            "output_port": "data",
            "name": "sale_refund",
            "df_name": "sale_refund"
        }
    ]
}
out = run(config, inputs, spark)
ctx["sql_1.result"] = out["result"]

In [0]:
"""
id: hub.default.output_2
template: output
templateVersion: 1.0.0
name: hub.default.output_2
position:
  x: 600
  y: 0
description:
  text: Save data to a table named output_2 in the hub catalog and default schema, overwriting existing content.
  hash: 4fc68022
previewCodeHash: ee54f5816f74fb8f
previewMode: "1000"
config:
  catalog: hub
  schema: "`default`"
  table_name: output_2
input:
  - node: sql_1
    input_port: data
    output_port: result
"""

# generated from the system
from typing import Dict, Any

def run(
    config: Dict[str, Any], inputs: Dict[str, Any], spark
) -> Dict[str, Any]:
    df = inputs["data"]
    catalog = config.get("catalog", "")
    schema = config.get("schema", "")
    table_name = config.get("table_name", "")

    if not table_name:
        raise ValueError("Output: 'table_name' is required")

    parts = [p for p in [catalog, schema, table_name] if p]
    full_name = ".".join(parts)

    df.write.mode("overwrite").saveAsTable(full_name)

    return {}

# generated from the system
ctx = globals().setdefault("ctx", {})
config = {
    "catalog": "hub",
    "schema": "`default`",
    "table_name": "output_2"
}
inputs = {
    "data": ctx["sql_1.result"]
}
out = run(config, inputs, spark)